# Legal Metrology ML: Exploratory Data Analysis (EDA)

**Project:** Legal Metrology ML Verification Engine  
**Dataset:** Synthetic Verification Records (Simulated Physics & Metrology Standards)  

> **DISCLAIMER:**  
> This notebook analyzes synthetic data generated for software architecture design and ML experimentation.  
> It does NOT represent statutory Indian Legal Metrology records.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root in sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.generate_dataset import generate_synthetic_dataset
from src.data.feature_engineering import engineer_features
from src.data.validate_dataset import validate_dataset_dataframe

## 1. Load and Validate Dataset

In [ ]:
data_path = project_root / "data" / "synthetic" / "synthetic_verifications.csv"
if data_path.exists():
    df = pd.read_csv(data_path)
else:
    df = generate_synthetic_dataset(num_rows=5000, seed=42)

df, report = validate_dataset_dataframe(df)
df = engineer_features(df)
print("Dataset Shape:", df.shape)
df.head()

## 2. Instrument Types & Accuracy Classes Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

df["instrument_type"].value_counts().plot(kind="barh", ax=ax1, color="#1f77b4")
ax1.set_title("Distribution by Instrument Type")
ax1.set_xlabel("Inspection Records Count")

df["accuracy_class"].value_counts().plot(kind="bar", ax=ax2, color="#ff7f0e")
ax2.set_title("Distribution by Accuracy Class (OIML R76 Classes)")
ax2.set_xlabel("Accuracy Class")
ax2.set_ylabel("Count")

plt.tight_layout()
plt.show()

## 3. Verification Outcomes (PASS vs FAIL)

In [ ]:
outcome_counts = df["verification_result"].value_counts()
print(outcome_counts)
plt.figure(figsize=(5, 5))
plt.pie(outcome_counts, labels=outcome_counts.index, autopct="%1.1f%%", colors=["#2ca02c", "#d62728"])
plt.title("Verification Result Balance")
plt.show()

## 4. Measurement Error Distributions vs. Verification Outcome

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for outcome, color in [("PASS", "green"), ("FAIL", "red")]:
    subset = df[df["verification_result"] == outcome]["maximum_absolute_error"]
    subset[subset < subset.quantile(0.95)].plot(kind="hist", bins=30, alpha=0.5, label=outcome, color=color, ax=ax)

ax.set_title("Maximum Absolute Error Distribution by Result")
ax.set_xlabel("Maximum Absolute Error (kg)")
ax.legend()
plt.show()